# HDBSCAN clustering

Clusters all feature titles using the shared similarity matrix from Part 3 of `final_report.ipynb`. This follows the same workflow as `6_dbscan.ipynb`, but HDBSCAN can find clusters with different densities and marks outliers as noise.

**Method:** convert similarity to distance, compare a small grid of HDBSCAN parameters, inspect the selected clusters, select by silhouette with noise points counted as singleton clusters, visualize with t-SNE, and check parameter sensitivity with grid heatmaps.

**Requirement:** HDBSCAN is included in scikit-learn 1.3 and later.

In [1]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

MIN_CLUSTER_SIZE_VALUES = [2, 3, 4, 5, 8, 12, 20]
MIN_SAMPLES_VALUES = [2, 3, 4, 5, 6, 8]
SEED = 0
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    attributes = data["titles"].tolist()

n = len(attributes)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"

distance_matrix = np.clip(1 - similarity, 0, 1)
np.fill_diagonal(distance_matrix, 0)

print(f"Loaded {n} attributes, matrix {similarity.shape}")

Loaded 445 attributes, matrix (445, 445)


## 1. Parameter search

`min_cluster_size` sets the smallest group HDBSCAN may return. `min_samples` controls how conservatively it treats points as noise; larger values usually label more points as noise.

Parameters are selected by the silhouette score computed on all attributes, with each noise point treated as its own singleton cluster (silhouette 0). This prevents a configuration from scoring well by discarding difficult attributes as noise. Ties are broken by higher coverage, then fewer clusters. Both methods search the same `min_samples` range; DBSCAN additionally searches `eps` and HDBSCAN `min_cluster_size`. HDBSCAN has no `eps` parameter.

The silhouette on non-noise points only is reported for reference but is not used for selection.

In [2]:
def noise_as_singletons(labels):
    relabeled = labels.copy()
    noise_indices = np.where(relabeled == -1)[0]
    start = relabeled.max() + 1
    relabeled[noise_indices] = np.arange(start, start + len(noise_indices))
    return relabeled


def evaluate(labels, D):
    is_noise = labels == -1
    n_noise = int(is_noise.sum())
    n_clusters = len(set(labels[~is_noise].tolist()))
    if n_clusters < 2:
        return None

    singleton_labels = noise_as_singletons(labels)
    if len(set(singleton_labels.tolist())) >= len(labels):
        return None
    score = silhouette_score(D, singleton_labels, metric="precomputed")

    non_noise_labels = labels[~is_noise]
    if n_clusters < len(non_noise_labels):
        silhouette_non_noise = silhouette_score(
            D[np.ix_(~is_noise, ~is_noise)], non_noise_labels, metric="precomputed"
        )
    else:
        silhouette_non_noise = float("nan")

    return {
        "score": float(score),
        "silhouette_non_noise": float(silhouette_non_noise),
        "coverage": float((~is_noise).mean()),
        "n_clusters": int(n_clusters),
        "n_noise": n_noise,
    }


def format_row(row, param_names):
    params = ", ".join(f"{name}={row[name]:g}" for name in param_names)
    return (
        f"{params} | "
        f"score={row['score']:.4f}, silhouette_non_noise={row['silhouette_non_noise']:.4f}, "
        f"clusters={row['n_clusters']}, coverage={100 * row['coverage']:.1f}%, noise={row['n_noise']}"
    )


def warn_if_at_grid_edge(best, grids):
    for name, grid in grids.items():
        if np.isclose(best[name], min(grid)) or np.isclose(best[name], max(grid)):
            print(f"Warning ({name}={best[name]:g}): Selected value is at the edge of the grid - consider extending the grid.")


PARAM_GRIDS = {"min_cluster_size": MIN_CLUSTER_SIZE_VALUES, "min_samples": MIN_SAMPLES_VALUES}

candidates = []
for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
    for min_samples in MIN_SAMPLES_VALUES:
        if min_samples > min_cluster_size:
            continue

        model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="precomputed",
            copy=True,
        )
        labels = model.fit_predict(distance_matrix)
        evaluation = evaluate(labels, distance_matrix)
        if evaluation is None:
            continue
        candidates.append({"min_cluster_size": min_cluster_size, "min_samples": min_samples, **evaluation})

if not candidates:
    raise ValueError("No valid HDBSCAN configuration produced at least 2 non-noise clusters.")

candidates = sorted(candidates, key=lambda r: (-r["score"], -r["coverage"], r["n_clusters"]))
best = candidates[0]

print("Top 10 parameter sets (score = silhouette with noise as singletons):")
for row in candidates[:10]:
    print(format_row(row, PARAM_GRIDS))

print("\nSelected configuration:")
print(format_row(best, PARAM_GRIDS))
warn_if_at_grid_edge(best, PARAM_GRIDS)

Top 10 parameter sets (score = silhouette with noise as singletons):
min_cluster_size=2, min_samples=2 | score=0.1778, silhouette_non_noise=0.2813, clusters=106, coverage=73.3%, noise=119
min_cluster_size=3, min_samples=2 | score=0.1103, silhouette_non_noise=0.2360, clusters=61, coverage=67.0%, noise=147
min_cluster_size=4, min_samples=2 | score=0.0725, silhouette_non_noise=0.2292, clusters=39, coverage=59.6%, noise=180
min_cluster_size=3, min_samples=3 | score=0.0520, silhouette_non_noise=0.2924, clusters=38, coverage=47.0%, noise=236
min_cluster_size=5, min_samples=2 | score=0.0473, silhouette_non_noise=0.2218, clusters=30, coverage=56.9%, noise=192
min_cluster_size=4, min_samples=4 | score=0.0149, silhouette_non_noise=0.2833, clusters=20, coverage=41.1%, noise=262
min_cluster_size=5, min_samples=4 | score=0.0147, silhouette_non_noise=0.2814, clusters=18, coverage=39.3%, noise=270
min_cluster_size=4, min_samples=3 | score=0.0140, silhouette_non_noise=0.2430, clusters=25, coverage=47.

## 2. Fit HDBSCAN and inspect groups

In [3]:
final_model = HDBSCAN(
    min_cluster_size=best["min_cluster_size"],
    min_samples=best["min_samples"],
    metric="precomputed",
    copy=True,
)
labels = final_model.fit_predict(distance_matrix)

sizes = Counter(labels.tolist())
cluster_labels = [label for label in sorted(sizes) if label != -1]
n_noise = sizes.get(-1, 0)

print(
    f"Clusters (excluding noise): {len(cluster_labels)} | "
    f"coverage: {100 * (len(labels) - n_noise) / len(labels):.1f}% | "
    f"noise points: {n_noise} / {len(labels)} ({100 * n_noise / len(labels):.1f}%)"
)
print(f"Cluster sizes (largest first): {[sizes[label] for label in sorted(cluster_labels, key=lambda label: -sizes[label])]}")

members_by_cluster = {
    label: [attributes[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in cluster_labels
}

for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    print(f"\nCluster {label} ({len(members)} attributes)")
    for member in members:
        print(f"  - {member}")

Clusters (excluding noise): 106 | coverage: 73.3% | noise points: 119 / 445 (26.7%)
Cluster sizes (largest first): [9, 8, 7, 7, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Cluster 105 (9 attributes)
  - Windproof Front Panel
  - Windproof protection
  - Windproof softshell 3-layer panels
  - Windtex Softshell 3-layer fabric
  - Windproof front panel
  - Windproof softshell fabric
  - Windproof front panels
  - Deep winter windproof protection
  - Windproof

Cluster 88 (8 attributes)
  - Merino wool fabric mix
  - Luxurious soft merino blend
  - Merino wool lining
  - Pure merino wool
  - Merino wool fabric blend
  - Durable Merino-Polyester Blend
  - Merino wool blend
  - Premium merino wool blend

Cluster 29 (7 attributes)
  -

## 3. Cluster visualization

t-SNE projects the precomputed distances (`1 - similarity`) to two dimensions for display only. The largest clusters get highlight colors, smaller clusters are grey, and noise points are shown as hollow markers.

In [4]:
coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    learning_rate="auto",
    random_state=SEED,
).fit_transform(distance_matrix)

PALETTE = ["#2f6b8a", "#e07a5f", "#3d9970", "#8e6cbb", "#d4a72c", "#d45d79", "#5086c1", "#8c6d5c"]
ranked_clusters = sorted(cluster_labels, key=lambda label: (-sizes[label], label))
highlighted_clusters = ranked_clusters[:HIGHLIGHT_GROUPS]
group_names = {label: f"Cluster {label}" for label in cluster_labels}

def add_points(indices, name, color, hollow=False, size=8):
    if len(indices) == 0:
        return

    marker = {"size": size, "color": color}
    if hollow:
        marker.update({"symbol": "circle-open", "line": {"color": color, "width": 1.5}})

    figure.add_trace(
        go.Scatter(
            x=coordinates[indices, 0],
            y=coordinates[indices, 1],
            mode="markers",
            name=name,
            marker=marker,
            text=[attributes[index] for index in indices],
            customdata=["Noise" if labels[index] == -1 else group_names[labels[index]] for index in indices],
            hovertemplate="<b>%{text}</b><br>%{customdata}<extra></extra>",
        )
    )

figure = go.Figure()
noise_indices = np.where(labels == -1)[0]
add_points(noise_indices, f"Noise ({len(noise_indices)})", "#b8bec5", hollow=True, size=7)

other_indices = np.array(
    [index for index, label in enumerate(labels) if label != -1 and label not in highlighted_clusters],
    dtype=int,
)
add_points(other_indices, "Other clusters", "#b8bec5", size=7)

for color, label in zip(PALETTE, highlighted_clusters):
    indices = np.where(labels == label)[0]
    add_points(indices, f"Cluster {label} ({sizes[label]})", color, size=10)

figure.update_layout(
    title=dict(text="Attribute clusters: HDBSCAN (t-SNE)", x=0),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", itemsizing="constant"),
    hoverlabel=dict(bgcolor="white"),
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

## 5. Evaluation

### 5.1 Internal quality

HDBSCAN marks some points as noise (`-1`). The selection score treats each noise point as its own singleton cluster (silhouette 0), so discarding attributes as noise does not inflate it. The silhouette on non-noise points only is shown for reference.

In [5]:
metrics = evaluate(labels, distance_matrix)

if metrics is None:
    print("Silhouette score not defined (need at least 2 non-noise clusters).")
else:
    print(f"Silhouette score (noise as singletons, selection score): {metrics['score']:.4f}")
    print(f"Silhouette score (non-noise only, reported only): {metrics['silhouette_non_noise']:.4f}")
    print(f"Clusters (excluding noise): {metrics['n_clusters']}")
    print(f"Coverage: {100 * metrics['coverage']:.1f}%")
    print(f"Noise points: {metrics['n_noise']} / {len(labels)}")

Silhouette score (noise as singletons, selection score): 0.1778
Silhouette score (non-noise only, reported only): 0.2813
Clusters (excluding noise): 106
Coverage: 73.3%
Noise points: 119 / 445


### 5.2 Parameter selection

These plots show how sensitive the result is to the chosen parameters, using the results already computed in the parameter search (no refitting).

- **Heatmaps** show the selection score, coverage and number of clusters for every configuration in the grid, with the same colour map in both notebooks. The red square marks the selected configuration. Blank cells produced fewer than 2 clusters or were skipped because `min_samples > min_cluster_size`. A selected cell inside a broad bright area means the choice is stable; an isolated peak, or one at the grid edge, means it depends on the exact values.
- **Sensitivity summary** lists the score and number of clusters one grid step away from the selected configuration in each direction. Small changes in score but large changes in cluster count mean the score alone does not pin down the granularity.

_Placeholder: HDBSCAN finds N clusters with the selected parameters, compared with k = … chosen for spectral clustering in `5_spectral.ipynb`._

In [6]:
SELECTED_COLOR = "#cc1f1a"
HEATMAP_COLORSCALE = "Viridis"
HEATMAP_METRICS = {
    "score": ("Score (silhouette, noise as singletons)", ".3f"),
    "coverage": ("Coverage (%)", ".1f"),
    "n_clusters": ("Clusters", ".0f"),
}


def grid_table(candidates, x_name, x_values, y_name, y_values):
    lookup = {(row[x_name], row[y_name]): row for row in candidates}
    table = {metric: np.full((len(y_values), len(x_values)), np.nan) for metric in HEATMAP_METRICS}
    for j, x in enumerate(x_values):
        for i, y in enumerate(y_values):
            row = lookup.get((x, y))
            if row is None:
                continue
            for metric in HEATMAP_METRICS:
                table[metric][i, j] = 100 * row[metric] if metric == "coverage" else row[metric]
    return table


def plot_param_heatmaps(table, x_name, x_values, y_name, y_values, best, method):
    x_labels = [f"{x:g}" for x in x_values]
    y_labels = [f"{y:g}" for y in y_values]
    figure = make_subplots(
        rows=len(HEATMAP_METRICS), cols=1,
        subplot_titles=[title for title, _ in HEATMAP_METRICS.values()],
        vertical_spacing=0.1,
    )
    for row_index, (metric, (title, fmt)) in enumerate(HEATMAP_METRICS.items(), start=1):
        domain = figure.get_subplot(row_index, 1).yaxis.domain
        figure.add_trace(
            go.Heatmap(
                z=table[metric], x=x_labels, y=y_labels,
                colorscale=HEATMAP_COLORSCALE, hoverongaps=False,
                colorbar=dict(len=domain[1] - domain[0], y=(domain[0] + domain[1]) / 2, thickness=12),
                hovertemplate=f"{x_name}=%{{x}}<br>{y_name}=%{{y}}<br>{title}: %{{z:{fmt}}}<extra></extra>",
            ),
            row=row_index, col=1,
        )
        figure.add_trace(
            go.Scatter(
                x=[f"{best[x_name]:g}"], y=[f"{best[y_name]:g}"], mode="markers",
                marker=dict(symbol="square-open", size=18, color=SELECTED_COLOR, line=dict(width=3)),
                name="Selected", showlegend=row_index == 1, hoverinfo="skip",
            ),
            row=row_index, col=1,
        )
        figure.update_xaxes(
            title_text=x_name if row_index == len(HEATMAP_METRICS) else None,
            type="category", tickangle=-45 if len(x_values) > 12 else 0, row=row_index, col=1,
        )
        figure.update_yaxes(title_text=y_name, type="category", row=row_index, col=1)
    figure.update_layout(
        title=dict(text=f"{method} parameter grid (blank = fewer than 2 clusters or not evaluated)", x=0),
        template="simple_white", width=1000, height=1000,
        legend=dict(orientation="h", x=1, xanchor="right", y=1.04),
    )
    show(figure)


def print_sensitivity(candidates, best, grids):
    names = list(grids)
    lookup = {tuple(row[name] for name in names): row for row in candidates}
    print(f"Selected: {format_row(best, grids)}")
    print("\nOne grid step away (score change vs selected):")
    for name, grid in grids.items():
        grid = list(grid)
        index = int(np.argmin([abs(value - best[name]) for value in grid]))
        for step in (-1, 1):
            if not 0 <= index + step < len(grid):
                print(f"  {name}: no {'lower' if step < 0 else 'higher'} value in grid")
                continue
            key = tuple(grid[index + step] if other == name else best[other] for other in names)
            params = ", ".join(f"{other}={value:g}" for other, value in zip(names, key))
            row = lookup.get(key)
            if row is None:
                print(f"  {params} | fewer than 2 clusters or not evaluated")
            else:
                print(
                    f"  {params} | score={row['score']:.4f} ({row['score'] - best['score']:+.4f}), "
                    f"clusters={row['n_clusters']}"
                )


def show(figure):
    try:
        figure.show(renderer="notebook_connected")
    except ValueError:
        display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))


grids = {"min_cluster_size": MIN_CLUSTER_SIZE_VALUES, "min_samples": MIN_SAMPLES_VALUES}
table = grid_table(candidates, "min_cluster_size", grids["min_cluster_size"], "min_samples", grids["min_samples"])
plot_param_heatmaps(table, "min_cluster_size", grids["min_cluster_size"], "min_samples", grids["min_samples"], best, "HDBSCAN")
print_sensitivity(candidates, best, grids)

Selected: min_cluster_size=2, min_samples=2 | score=0.1778, silhouette_non_noise=0.2813, clusters=106, coverage=73.3%, noise=119

One grid step away (score change vs selected):
  min_cluster_size: no lower value in grid
  min_cluster_size=3, min_samples=2 | score=0.1103 (-0.0675), clusters=61
  min_samples: no lower value in grid
  min_cluster_size=2, min_samples=3 | fewer than 2 clusters or not evaluated
